## Prep for Discrete-Time Hazard

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [2]:
# load prepared discrete time hazard data
PATH = r"..\data\cleaned\discrete_time_hazard_data\startup_seriesA_person_period_quarterly_final_all.csv"
df = pd.read_csv(PATH, parse_dates=["period_start","seed_date"])

### Clean / transform covariates

In [5]:
# winsorize 1% tails
low, high = df["seed_amount_usd"].quantile([0.01,0.99])
df["seed_amount_usd_w"] = df["seed_amount_usd"].clip(lower=low, upper=high)

# log transform
df["log_seed_amt"] = np.log1p(df["seed_amount_usd_w"])

In [6]:
# process num_partner_investors
df["num_partner_investors"] = pd.to_numeric(df["num_partner_investors"], errors="coerce")

# log transform with fillna(0)
df["log_npi"] = np.log1p(df["num_partner_investors"].fillna(0))

### Fixed Effects and duration

In [7]:
# --- Duration bins (5 bins) instead of 20 dummies ---
bins = [-1, 2, 5, 8, 12, 10**9]
labels = ["dur_b0_2","dur_b3_5","dur_b6_8","dur_b9_12","dur_b13p"]
df["dur_bin"] = pd.cut(df["duration_q"], bins=bins, labels=labels)
dur_d = pd.get_dummies(df["dur_bin"], drop_first=True)  # base = 0–2

In [8]:
# --- Cohort FE by YEAR (not quarter) ---
df["seed_year"] = df["seed_date"].dt.year
cohort_d = pd.get_dummies(df["seed_year"].astype("Int64"), prefix="cohortY", drop_first=True)

#### Sector / Metro Fixed Effects

In [ ]:
# create dummies for sector and metro_regions
X_list = []
if "sector" in df.columns:
    X_list.append(pd.get_dummies(df["sector"].astype(str), prefix="sector", drop_first=True)) # sector dummies
if "metro_region" in df.columns:
    metro = df["metro_region"].astype(str)
    X_list.append(pd.get_dummies(metro, prefix="metro", drop_first=True)) # metro region dummies

### Core Regressors

In [ ]:
core = [
    "shock_25bp_units","shock25_lag1","shock25_lag2","shock25_lag3","shock25_lag4","shock25_lag5",
    "nasdaqcom_ccr","t10y2y",
    "log_seed_amt","log_npi"
]

# combine all X variables
X_core = df[core]

In [11]:
# final X and y
X = pd.concat([X_core, dur_d, cohort_d] + X_list, axis=1)
y = df["event_seriesA"].astype(int)

In [12]:
# drop rows with missing values in X or y
data = pd.concat([y, X], axis=1).dropna()

# cleaned y and X
y_clean = data["event_seriesA"].astype(int)

# final X with constant
X_clean = sm.add_constant(data.drop(columns=["event_seriesA"]), has_constant="add")

In [13]:
# Cluster by quarter
groups = df.loc[data.index, "period_start"].dt.to_period("Q").astype(str)

In [14]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# --- 0) Make sure types are numeric ---
# (convert any bool dummies to 0/1; keep float for GLM)
bool_cols = X_clean.select_dtypes(include=["bool"]).columns
X_clean[bool_cols] = X_clean[bool_cols].astype(np.uint8)
X_clean = X_clean.astype(float)
y_clean = y_clean.astype(int)

# --- 1) Drop zero-variance / constant columns ---
const_cols = X_clean.columns[X_clean.nunique() <= 1].tolist()
if const_cols:
    print("Dropping constant cols:", const_cols)
    X_clean = X_clean.drop(columns=const_cols)

# --- 2) Drop exact duplicate columns ---
dup_mask = X_clean.T.duplicated()
if dup_mask.any():
    dups = X_clean.columns[dup_mask].tolist()
    print("Dropping duplicate cols:", dups)
    X_clean = X_clean.loc[:, ~dup_mask]

# --- 3) Drop dummies that perfectly separate the outcome (complete separation) ---
sep_cols = []
for c in X_clean.columns:
    # only check 0/1-ish columns to keep it cheap
    vals = X_clean[c].dropna().unique()
    if set(np.round(vals, 6)).issubset({0.0, 1.0}):
        s = X_clean[c] == 1
        if s.any() and (~s).any():
            y1 = y_clean[s]
            y0 = y_clean[~s]
            if (y1.nunique() == 1 and y0.nunique() == 1 and y1.iloc[0] != y0.iloc[0]) or \
               (y1.nunique() == 1 and (y1.iloc[0] in [0,1]) and (y_clean[s].mean() in [0,1])):
                sep_cols.append(c)
        elif s.any() and (y_clean[s].nunique() == 1):
            sep_cols.append(c)
if sep_cols:
    print("Dropping separation cols:", sep_cols)
    X_clean = X_clean.drop(columns=sep_cols)

# --- 4) Make sure we still have full-rank numerically (QR check) ---
# If SciPy is available, pivoted QR is best; this is a simple SVD-based filter.
U, svals, Vt = np.linalg.svd(X_clean.values, full_matrices=False)
rank = (svals > 1e-10).sum()
if rank < X_clean.shape[1]:
    # keep the first 'rank' right-singular vectors (largest svals)
    keep_idx = np.argsort(-svals)[:rank]   # indices sorted by descending svals
    # map back to columns via Vt rows
    # get importance of each column ~ squared loading in leading singular vectors
    col_score = (Vt[:rank, :]**2).sum(axis=0)
    order = np.argsort(-col_score)[:rank]
    print(f"Reducing to full rank: {X_clean.shape[1]} → {rank} columns")
    X_clean = X_clean.iloc[:, order]

# --- 5) Add constant and fit (cluster by quarter) ---
X_clean = sm.add_constant(X_clean, has_constant="add")
assert len(groups) == len(y_clean)

model = sm.GLM(y_clean, X_clean,
               family=sm.families.Binomial(link=sm.families.links.cloglog()))
res = model.fit(cov_type="cluster", cov_kwds={"groups": groups})
print(res.summary())


Dropping constant cols: ['const']
Dropping separation cols: ['cohortY_2023']
                 Generalized Linear Model Regression Results                  
Dep. Variable:          event_seriesA   No. Observations:                14520
Model:                            GLM   Df Residuals:                    14494
Model Family:                Binomial   Df Model:                           25
Link Function:                cloglog   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1839.8
Date:                Wed, 10 Dec 2025   Deviance:                       3679.6
Time:                        16:01:48   Pearson chi2:                 1.45e+04
No. Iterations:                     8   Pseudo R-squ. (CS):            0.02590
Covariance Type:              cluster                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------

c:\Users\Ricardo\anaconda3\Lib\site-packages\statsmodels\genmod\families\links.py:13: FutureWarning: The cloglog link alias is deprecated. Use CLogLog instead. The cloglog link alias will be removed after the 0.15.0 release.
  warnings.warn(
